# 09 -- Сравнение моделей и ансамбли

Собираю результаты предыдущих ноутбуков. Проверяю простое смешивание рангов и stacking.
Для метамодели делю dev на calibration и comparison. Базовые модели обучены на rank_train.
Весь dev уже участвовал в выборе базовых моделей, поэтому comparison не называю новым независимым тестом.
Из проверенных вариантов каждой модели беру лучший по dev: 300 деревьев у CatBoost
и QuerySoftMax, три эпохи у MLP.

In [1]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from IPython.display import display

en_root = Path.cwd() if (Path.cwd() / 'notebooks').exists() else Path.cwd().parent
import nbformat

# Подключаю только ячейки с функциями; расчеты других ноутбуков не запускаю.
for en_file in ['02_feature_engineering.ipynb']:
    for en_cell in nbformat.read(en_root / 'notebooks' / en_file, as_version=4).cells:
        if 'definitions' in en_cell.metadata.get('tags', []):
            exec(en_cell.source, globals())
en_out = en_root / 'outputs'
en_seed = 143
en_scores = pd.read_parquet(en_out / 'cache/scores_catboost.parquet')
for en_family in ['ln', 'rk', 'ml']:
    en_other = pd.read_parquet(en_out / f'cache/scores_{en_family}.parquet')
    en_before = len(en_scores)
    assert len(en_other) == en_before, f'Другой набор кандидатов: {en_family}'
    en_scores = en_scores.merge(
        en_other, on=['query_id', 'pos'], how='inner', validate='one_to_one'
    )
    assert len(en_scores) == en_before, f'Потеряны пары: {en_family}'

en_queries = pd.read_parquet(en_out / 'cache/eval_queries.parquet').query(
    "split == 'dev'"
)
en_items = pd.read_parquet(en_out / 'cache/items.parquet', columns=['item_id'])
en_truth_all = json.loads((en_out / 'cache/truth.json').read_text())
en_truth = {q: en_truth_all[q] for q in en_queries.query_id}
en_warm_share = json.loads((en_out / 'models/hybrid.json').read_text())['warm_share']

In [2]:
en_models_summary = pd.concat(
    [
        pd.read_csv(en_out / f'validation/models_{family}.csv')
        for family in ['cb', 'ln', 'rk', 'ml']
    ],
    ignore_index=True,
).sort_values('objective', ascending=False)
en_models_summary.to_csv(en_out / 'validation/model_comparison.csv', index=False)
display(en_models_summary)

,model,cold,warm,objective,fit_seconds
1,extended_300,0.800472,0.815679,0.806172,81.079984
0,basic_300,0.800639,0.809536,0.803973,0.000000
2,extended_500,0.792472,0.812179,0.799858,81.079984
6,mlp_3,0.795472,0.805345,0.799173,2.189645
7,mlp_6,0.796972,0.800155,0.798165,4.798435
3,linear,0.787806,0.804345,0.794005,9.069299
5,ranker_300,0.786806,0.802071,0.792527,333.108101
4,ranker_150,0.775472,0.793238,0.782131,333.108101


## Разделение для метамодели

In [3]:
en_cal_ids = (
    en_queries.groupby('regime', group_keys=False)
    .sample(frac=0.5, random_state=en_seed)
    .query_id
)

In [4]:
en_queries = en_queries.assign(
    meta_part=np.where(
        en_queries.query_id.isin(en_cal_ids), 'calibration', 'comparison'
    )
)
assert not set(en_cal_ids) & set(
    en_queries.loc[en_queries.meta_part.eq('comparison'), 'query_id']
)
en_queries[['query_id', 'regime', 'meta_part']].to_csv(
    en_out / 'validation/meta_split.csv', index=False
)
en_models = ['basic_300', 'extended_300', 'ranker_300', 'mlp_3']
en_rank_features = pd.DataFrame(index=en_scores.index)
for en_name in en_models:
    en_ordered = en_scores.sort_values(
        ['query_id', en_name, 'pos'], ascending=[True, False, True]
    )
    en_rank = en_ordered.groupby('query_id').cumcount() + 1
    en_rank_features[en_name] = (31 / (30 + en_rank)).reindex(en_scores.index)
en_results = []

In [5]:
def en_evaluate(name, values):
    predictions = predictions_from_frame(
        en_scores.assign(score=values), en_items, score='score', k=100
    )
    metrics = recall_table(predictions, en_truth).merge(
        en_queries[['query_id', 'regime', 'meta_part']], on='query_id'
    )
    for part, group in metrics.groupby('meta_part'):
        scores = group.groupby('regime')['recall@50'].mean()
        en_results.append(
            {
                'variant': name,
                'part': part,
                'cold': scores['cold'],
                'warm': scores['warm'],
                'objective': (1 - en_warm_share) * scores['cold']
                + en_warm_share * scores['warm'],
            }
        )
    return metrics

## Смешивание рангов

Проверяю небольшую сетку весов. Выбираю вес только по calibration, затем показываю comparison.
Скоры моделей не складываю напрямую: у Logloss, QuerySoftMax и MLP разные шкалы.

In [6]:
en_values = {}
for en_model in ['extended_300', 'ranker_300', 'mlp_3']:
    for en_alpha in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
        en_name = f'{en_model}_{en_alpha:g}'
        en_values[en_name] = (
            1 - en_alpha
        ) * en_rank_features.basic_300 + en_alpha * en_rank_features[en_model]
        en_evaluate(en_name, en_values[en_name])
en_blends = pd.DataFrame(en_results)
en_selected = (
    en_blends.query("part == 'calibration'")
    .sort_values(['objective', 'variant'], ascending=[False, True])
    .iloc[0]
    .variant
)
display(en_blends[en_blends.variant.eq(en_selected)])

,variant,part,cold,warm,objective
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587


## Логистическая метамодель

Вход -- четыре ранговых признака. Базовые модели не обучались на dev, поэтому метамодель
получает отложенные предсказания. Calibration использую для fit, comparison -- только для диагностики.
Сравниваю с простым blending: дополнительный уровень должен оправдать себя.

In [7]:
en_target = np.array(
    [
        int(en_items.item_id.iloc[p] in en_truth[q])
        for q, p in zip(en_scores.query_id, en_scores.pos)
    ]
)
en_cal_mask = en_scores.query_id.isin(en_cal_ids).to_numpy()
en_labels = en_scores[['query_id']].assign(target=en_target)
en_npos = en_labels.groupby('query_id').target.transform('sum')
en_n = en_labels.groupby('query_id').target.transform('size')
en_fit_mask = en_cal_mask & en_npos.gt(0).to_numpy()
en_weights = np.where(
    en_target == 1, 0.5 / en_npos.clip(lower=1), 0.5 / (en_n - en_npos).clip(lower=1)
)
en_stack = LogisticRegression(C=1.0, max_iter=300, random_state=en_seed)
en_stack.fit(
    en_rank_features.loc[en_fit_mask],
    en_target[en_fit_mask],
    sample_weight=en_weights[en_fit_mask] / en_weights[en_fit_mask].mean(),
)
en_stack_metrics = en_evaluate('stacking', en_stack.decision_function(en_rank_features))
en_results = pd.DataFrame(en_results)
en_results.to_csv(en_out / 'validation/ensemble_experiments.csv', index=False)
display(
    en_results[en_results.variant.isin([en_selected, 'stacking', 'extended_300_0'])]
)

,variant,part,cold,warm,objective
0,extended_300_0,calibration,0.806667,0.821571,0.812253
1,extended_300_0,comparison,0.794611,0.797500,0.795694
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587
36,stacking,calibration,0.812667,0.824571,0.817129
37,stacking,comparison,0.792944,0.804000,0.797088


## Проверка выбранного веса

Calibration выбрал 75% расширенной модели. На comparison этот вариант дал 0.79159
против 0.79569 исходной модели. Не готовлю его к отправке: локальный выбор веса не подтвердился
даже на второй половине dev. Stacking дал 0.79709 -- всего +0.00031 к одиночной расширенной модели на comparison; этого мало для дополнительной
метамодели с учетом повторного использования dev. Сохраняю эксперимент, финальную схему не усложняю.

In [8]:
en_cat_choices = en_blends[
    en_blends.part.eq('calibration') & en_blends.variant.str.startswith('extended_300_')
]
en_cat_choice = (
    en_cat_choices.sort_values(['objective', 'variant'], ascending=[False, True])
    .iloc[0]
    .variant
)
en_alpha = float(en_cat_choice.rsplit('_', 1)[1])
en_comparison = en_blends[en_blends.part.eq('comparison')].set_index('variant')
en_delta = float(
    en_comparison.loc[en_cat_choice, 'objective']
    - en_comparison.loc['extended_300_0', 'objective']
)
write_json(
    {
        'extended_alpha': en_alpha,
        'chosen_on': 'dev_calibration',
        'comparison_delta': en_delta,
        'recommended': False,
        'reason': 'Выбранный вес ухудшил comparison; stacking усложняет решение без убедительного выигрыша',
        'seed': en_seed,
        'official_score': None,
    },
    en_out / 'models/extended_blend.json',
)
display(en_blends[en_blends.variant.eq(en_cat_choice)])

,variant,part,cold,warm,objective
8,extended_300_0.75,calibration,0.811667,0.828857,0.818110
9,extended_300_0.75,comparison,0.785944,0.801000,0.791587


## Итоговый файл

В текущем сравнении смесь не оказалась устойчивее одиночного CatBoost.
Сохраняю его результат как answer.csv и проверяю ID по исходному benchmark.
Никакой старый CSV в расчет не входит.

In [9]:
en_queries_raw = pd.read_parquet(
    data_dir() / 'benchmark_queries.parquet', columns=['query_id']
)
en_items_raw = pd.read_parquet(
    data_dir() / 'benchmark_items.parquet', columns=['item_id']
)
en_source = en_out / 'submissions/answer_catboost.csv'
en_target_path = en_out / 'submissions/answer.csv'
en_target_path.write_bytes(en_source.read_bytes())
en_check = validate_submission(
    en_target_path, en_queries_raw.query_id.tolist(), en_items_raw.item_id.tolist()
)
write_json(en_check, en_out / 'submissions/validation.json')
display(en_check)

{'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}

## Проверка валидатора

Проверяю, что неверный формат останавливает отправку: повторный ID, лишняя колонка,
неизвестное объявление, неправильный разделитель и пропущенный запрос.

In [10]:
import csv
import tempfile

with tempfile.TemporaryDirectory() as en_temp:
    en_bad_path = Path(en_temp) / 'invalid.csv'
    en_qid = en_queries_raw.query_id.iloc[0]
    en_iid = en_items_raw.item_id.iloc[0]
    en_cases = [
        [['query_id', 'answer'], [en_qid, en_iid + ' ' + en_iid]],
        [['query_id', 'answer', 'index'], [en_qid, en_iid, '0']],
        [['query_id', 'answer'], [en_qid, 'INVALID_ITEM']],
        [['query_id', 'answer'], [en_qid, en_iid + '  ']],
        [['query_id', 'answer']],
    ]
    for en_case in en_cases:
        with en_bad_path.open('w', encoding='utf-8', newline='') as en_stream:
            csv.writer(en_stream).writerows(en_case)
        try:
            validate_submission(en_bad_path, [en_qid], [en_iid])
        except ValueError:
            continue
        raise AssertionError(f'Валидатор пропустил ошибку: {en_case}')
print('Ошибочные файлы отклонены')

Ошибочные файлы отклонены


## Результат отправки

Итоговый answer.csv получил Recall@50 = 0.822587 на площадке.
Сохраняю модель и параметры без изменений. Хеш файла и запись результата находятся
в outputs/submissions/submission_result.json. Этот результат не смешиваю с локальными метриками.